# Fire vs Caribou: re-runnable scorecard

Updates Canada's boreal caribou habitat scorecard (percent of each range undisturbed, federal method) past the 2020 federal map, through the latest complete fire season in the National Burned Area Composite.

**Re-running next year:** set `UPDATE_YEAR` in `caribou/config.py` to the new fire season, delete `data/raw/nbac` and `data/interim/pixels`, and run every cell. Everything else follows.

Works locally or in Google Colab (first cell installs the package). The full download is about 2.5 GB; the pixel step takes roughly 10-20 minutes at 100 m on a laptop.

In [ ]:
# In Colab: clone and install
# !git clone <this repository> caribou && cd caribou && pip install -e .
import json, pandas as pd
from caribou import config as C
pd.set_option("display.max_rows", 60, "display.precision", 3)

## 1. Download and prepare the open data

In [ ]:
!python -m caribou fetch prepare

## 2. Rasterize each range once (pixel-class tables) and compute exact vector shares

In [ ]:
!python -m caribou pixels vector --res 100 --alt-res 250

## 3. Scorecard, fire vs industry split, recovery clock, 2030 forecast

In [ ]:
!python -m caribou analyze figures
summary = json.loads((C.TABLES / "summary.json").read_text()); summary

In [ ]:
sc = pd.read_csv(C.TABLES / "scorecard.csv")
cols = ["CODE", "name", "threshold", "official_2020", "central_2025", "band_lo", "band_hi",
        "human_2025", "burned_2021_2025", "status_2025", "crossing"]
sc[cols].sort_values("central_2025")

### Reproduction of the official numbers (2015 and 2020 editions)

In [ ]:
rep = sc[["CODE", "name", "official_2015", "rebuild_2015_cnfdb", "official_2020", "rebuild_2020_cnfdb",
          "agree_2020_cnfdb", "unexplained"] + [c for c in sc if c.startswith("vec_")]]
rep

### Recovery clock and 2030 outlook

In [ ]:
rec = pd.read_csv(C.TABLES / "recovery.csv"); fc = pd.read_csv(C.TABLES / "forecast.csv")
rec[["CODE", "name", "recovery_class_official_adj", "recovery_year_official_adj", "ceiling_official_adj", "restore_km2"]]

In [ ]:
fc[["CODE", "name", "undisturbed_2025", "no_new_disturbance_2030", "p_below", "p_below_recent", "median_2030"]]

## 4. Figures

In [ ]:
from IPython.display import Image, display
for f in sorted(C.FIGURES.glob("fig*.png")):
    display(Image(filename=str(f), width=800))

## 5. Detector (Earth Engine) - optional

Needs `earthengine authenticate` once and an Earth Engine cloud project.

```
!python -m caribou.detector labels
!python -m caribou.detector sample --kind both --project <project>
!python -m caribou.detector evaluate
!python -m caribou.detector apply --kind emb --project <project>
!python -m caribou.detector postprocess
!python -m caribou pixels analyze figures --new-human detector_new_buffered
```